# Extração SIH/SUS — Dados de Internações Hospitalares para o projeto Conecta Saúde

Este notebook documenta a primeira fonte de dados do projeto: **SIH/SUS — Sistema de Informações Hospitalares do SUS**.

O objetivo é mostrar:

1. **quais dados serão extraídos**;
2. **qual endpoint público será usado**;
3. **como realizar a extração com Python**;
4. **quais campos são úteis para o projeto**;
5. **quais perguntas de negócio podem ser respondidas com esses dados**.

> Observação importante: o SIH/SUS não é consumido, na prática, como uma API REST tradicional com filtros por parâmetros. A forma pública mais usada para ingestão analítica é o download dos arquivos mensais em formato **DBC** no servidor público do DATASUS. Neste notebook, isso será tratado como **endpoint público de arquivos**. Para facilitar a leitura em Python, também é usado o pacote `pysus`, que encapsula o acesso aos microdados do DATASUS.

## 1. Papel do SIH/SUS no projeto

Dentro do projeto de **Painel Inteligente de Pressão Assistencial e Priorização Hospitalar**, o SIH/SUS será a fonte principal para medir **demanda hospitalar**.

A base será usada para analisar:

- volume de internações;
- município de residência do paciente;
- município de atendimento/internação;
- hospital ou estabelecimento executante, identificado pelo código CNES;
- período de competência;
- datas de internação e saída;
- dias de permanência;
- procedimento realizado;
- diagnóstico principal;
- óbito hospitalar;
- valores aprovados/pagos pelo SUS;
- complexidade e caráter da internação.

Na arquitetura do projeto, o SIH/SUS deve ser combinado posteriormente com:

- **CNES**, para cruzar demanda hospitalar com capacidade instalada, leitos e tipo de estabelecimento;
- **IBGE**, para normalizar os indicadores por população municipal;
- **SIGTAP**, para traduzir e classificar códigos de procedimentos em nomes e grupos compreensíveis para gestores.

## 2. Endpoint público de conexão

Para os dados consolidados de AIH reduzida, a estrutura de arquivos do SIH/SUS segue o padrão:

```text
https://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/RD{UF}{AAMM}.dbc
```

Exemplo para São Paulo, janeiro de 2024:

```text
https://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/RDSP2401.dbc
```

Onde:

| Parte | Significado |
|---|---|
| `RD` | Tipo de arquivo: AIH Reduzida, usado para análise consolidada de internações |
| `SP` | Unidade Federativa |
| `24` | Ano com dois dígitos: 2024 |
| `01` | Mês: janeiro |
| `.dbc` | Formato comprimido usado pelo DATASUS |

Para o MVP, a recomendação é começar com poucos meses e poucas UFs, validar o pipeline e depois ampliar o recorte temporal.

In [ ]:
# Instalação das dependências
# Execute esta célula no Colab/Jupyter caso os pacotes ainda não estejam instalados.

%pip install -q pandas pyarrow requests pysus

## 3. Parâmetros iniciais da extração

Abaixo ficam os parâmetros que controlam a extração.

Para um primeiro teste, recomenda-se começar por uma UF menor, como `AC`, para reduzir o tempo de download. Depois, basta trocar para `SP`, `RJ`, `MG` ou percorrer uma lista de UFs.

O grupo pode ajustar:

- `UF`: unidade federativa;
- `ANO`: ano de competência;
- `MESES`: lista de meses;
- `GRUPO`: tipo de arquivo do SIH/SUS. Para o projeto, usamos `RD`, que representa AIH reduzida.

In [ ]:
from pathlib import Path
from typing import Iterable
import pandas as pd
import requests

# Parâmetros de teste
UF = "SP"      # Ex.: "SP", "RJ", "MG", "AC"
ANO = 2024
MESES = [1]   # Ex.: [1, 2, 3] para janeiro, fevereiro e março
GRUPO = "RD"  # RD = AIH reduzida

BASE_URL_SIH_RD = "https://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/"
DIRETORIO_RAW = Path("dados/raw/sih_rd")
DIRETORIO_TRATADO = Path("dados/tratado/sih_rd")

DIRETORIO_RAW.mkdir(parents=True, exist_ok=True)
DIRETORIO_TRATADO.mkdir(parents=True, exist_ok=True)

## 4. Geração do link do arquivo DBC

Esta função monta automaticamente o link de conexão para qualquer UF, ano e mês.

Exemplo:

```python
montar_url_sih_rd("SP", 2024, 1)
```

Retorna:

```text
https://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Dados/RDSP2401.dbc
```

In [ ]:
def montar_url_sih_rd(uf: str, ano: int, mes: int) -> str:
    """
    Monta a URL pública do arquivo RD do SIH/SUS no servidor DATASUS.

    Parâmetros
    ----------
    uf : str
        Sigla da Unidade Federativa. Ex.: 'SP', 'RJ', 'MG'.
    ano : int
        Ano com quatro dígitos. Ex.: 2024.
    mes : int
        Mês de 1 a 12.

    Retorno
    -------
    str
        URL pública do arquivo DBC.
    """
    uf = uf.upper().strip()
    aa = str(ano)[-2:]
    mm = str(mes).zfill(2)
    arquivo = f"RD{uf}{aa}{mm}.dbc"
    return f"{BASE_URL_SIH_RD}{arquivo}"


# Exemplo de link gerado
for mes in MESES:
    print(montar_url_sih_rd(UF, ANO, mes))

## 5. Verificação de disponibilidade do arquivo

Antes de baixar o dado, é útil validar se o arquivo existe no endpoint público.

Alguns meses recentes podem ainda não estar disponíveis ou podem estar em versão preliminar. Se o status HTTP retornar `200`, o arquivo está acessível.

In [ ]:
def verificar_disponibilidade(url: str, timeout: int = 30) -> dict:
    """Verifica se o arquivo existe no endpoint público."""
    try:
        resposta = requests.head(url, timeout=timeout, allow_redirects=True)
        return {
            "url": url,
            "status_code": resposta.status_code,
            "disponivel": resposta.status_code == 200,
            "content_length_bytes": resposta.headers.get("Content-Length"),
            "content_type": resposta.headers.get("Content-Type"),
        }
    except requests.RequestException as erro:
        return {
            "url": url,
            "status_code": None,
            "disponivel": False,
            "erro": str(erro),
        }


disponibilidade = pd.DataFrame([
    verificar_disponibilidade(montar_url_sih_rd(UF, ANO, mes))
    for mes in MESES
])

disponibilidade

## 6. Extração com PySUS

O pacote `pysus` facilita a leitura dos microdados públicos do SUS e retorna os dados como `DataFrame`.

A função abaixo baixa os dados do SIH/SUS para UF, ano e mês informados.

> Caso a sua versão do `pysus` não aceite o argumento `group="RD"`, o código tenta executar novamente sem esse argumento.

In [ ]:
def extrair_sih_pysus(uf: str, ano: int, meses: Iterable[int], grupo: str = "RD") -> pd.DataFrame:
    """
    Extrai dados do SIH/SUS usando a API simplificada do PySUS.

    Parâmetros
    ----------
    uf : str
        Sigla da UF.
    ano : int
        Ano de competência.
    meses : Iterable[int]
        Lista de meses.
    grupo : str
        Grupo do SIH. Para AIH reduzida, usar 'RD'.

    Retorno
    -------
    pandas.DataFrame
        Dados brutos extraídos do SIH/SUS.
    """
    from pysus.api import sih

    uf = uf.upper().strip()
    meses = list(meses)

    try:
        df = sih(state=uf, year=ano, month=meses, group=grupo)
    except TypeError:
        # Fallback para versões em que group é opcional/indisponível na assinatura.
        df = sih(state=uf, year=ano, month=meses)

    # Garante retorno como pandas DataFrame, caso alguma versão retorne outro objeto compatível.
    if not isinstance(df, pd.DataFrame):
        try:
            df = df.to_pandas()
        except AttributeError:
            df = pd.DataFrame(df)

    return df


df_sih_raw = extrair_sih_pysus(UF, ANO, MESES, GRUPO)

print(f"Linhas extraídas: {df_sih_raw.shape[0]:,}")
print(f"Colunas extraídas: {df_sih_raw.shape[1]:,}")

df_sih_raw.head()

## 7. Quais dados serão extraídos

Os arquivos RD do SIH/SUS podem variar levemente conforme ano e layout, mas para o projeto os principais campos de interesse são:

| Campo | Uso no projeto |
|---|---|
| `ANO_CMPT` | Ano de competência da produção |
| `MES_CMPT` | Mês de competência da produção |
| `UF_ZI` | UF/município de processamento |
| `MUNIC_RES` | Município de residência do paciente |
| `MUNIC_MOV` | Município de movimentação/internação |
| `CNES` | Código do estabelecimento de saúde |
| `N_AIH` | Identificador da AIH, usado para contar internações |
| `DT_INTER` | Data de internação |
| `DT_SAIDA` | Data de saída |
| `DIAS_PERM` | Dias de permanência |
| `PROC_REA` | Procedimento realizado |
| `PROC_SOLIC` | Procedimento solicitado |
| `DIAG_PRINC` | Diagnóstico principal, em CID-10 |
| `SEXO` | Sexo do paciente |
| `IDADE` | Idade do paciente |
| `COD_IDADE` | Unidade de medida da idade |
| `MORTE` | Indicador de óbito hospitalar |
| `VAL_TOT` | Valor total aprovado/pago |
| `VAL_SH` | Valor de serviços hospitalares |
| `VAL_SP` | Valor de serviços profissionais |
| `VAL_UTI` | Valor relacionado à UTI |
| `COMPLEX` | Complexidade do atendimento |
| `CAR_INT` | Caráter da internação |
| `ESPEC` | Especialidade do leito/atendimento |

A célula abaixo mostra as colunas efetivamente disponíveis no arquivo extraído.

In [ ]:
# Lista completa de colunas presentes no arquivo extraído
pd.DataFrame({
    "coluna": df_sih_raw.columns,
    "tipo_detectado": [str(df_sih_raw[col].dtype) for col in df_sih_raw.columns]
})

## 8. Seleção dos campos úteis para o MVP

A célula abaixo cria uma base menor, contendo somente as colunas diretamente ligadas às perguntas do projeto.

Essa base tratada é mais adequada para carregar no banco analítico, construir dashboards e fazer cruzamentos com CNES e IBGE.

In [ ]:
COLUNAS_INTERESSE = [
    "ANO_CMPT",
    "MES_CMPT",
    "UF_ZI",
    "MUNIC_RES",
    "MUNIC_MOV",
    "CNES",
    "N_AIH",
    "DT_INTER",
    "DT_SAIDA",
    "DIAS_PERM",
    "PROC_SOLIC",
    "PROC_REA",
    "DIAG_PRINC",
    "SEXO",
    "IDADE",
    "COD_IDADE",
    "MORTE",
    "VAL_TOT",
    "VAL_SH",
    "VAL_SP",
    "VAL_UTI",
    "COMPLEX",
    "CAR_INT",
    "ESPEC",
]

colunas_existentes = [col for col in COLUNAS_INTERESSE if col in df_sih_raw.columns]
colunas_ausentes = [col for col in COLUNAS_INTERESSE if col not in df_sih_raw.columns]

df_sih = df_sih_raw[colunas_existentes].copy()

print("Colunas selecionadas:")
print(colunas_existentes)

print("\nColunas não encontradas neste recorte/layout:")
print(colunas_ausentes)

df_sih.head()

## 9. Tratamento básico dos tipos de dados

Os dados do DATASUS costumam chegar como texto. Para análise, é necessário converter:

- competência para data;
- datas de internação e saída;
- valores monetários;
- dias de permanência;
- idade;
- indicador de óbito.

In [ ]:
def converter_colunas_sih(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica conversões básicas nos campos mais usados do SIH/SUS."""
    df = df.copy()

    # Competência
    if {"ANO_CMPT", "MES_CMPT"}.issubset(df.columns):
        ano = df["ANO_CMPT"].astype(str).str.zfill(4)
        mes = df["MES_CMPT"].astype(str).str.zfill(2)
        df["COMPETENCIA"] = pd.to_datetime(ano + "-" + mes + "-01", errors="coerce")

    # Datas no padrão AAAAMMDD
    for col in ["DT_INTER", "DT_SAIDA"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col].astype(str), format="%Y%m%d", errors="coerce")

    # Campos numéricos
    campos_numericos = [
        "DIAS_PERM",
        "IDADE",
        "MORTE",
        "VAL_TOT",
        "VAL_SH",
        "VAL_SP",
        "VAL_UTI",
    ]

    for col in campos_numericos:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


df_sih = converter_colunas_sih(df_sih)

df_sih.info()

## 10. Dicionário analítico dos campos selecionados

A célula abaixo gera um pequeno dicionário de dados do recorte utilizado no projeto.

In [ ]:
DICIONARIO_CAMPOS = {
    "COMPETENCIA": "Data de competência da produção hospitalar.",
    "ANO_CMPT": "Ano da competência.",
    "MES_CMPT": "Mês da competência.",
    "UF_ZI": "Unidade de processamento/gestão da informação.",
    "MUNIC_RES": "Município de residência do paciente.",
    "MUNIC_MOV": "Município onde ocorreu a internação/movimentação.",
    "CNES": "Código do estabelecimento de saúde. Será chave para cruzamento com CNES.",
    "N_AIH": "Número da Autorização de Internação Hospitalar. Pode ser usado para contagem de internações.",
    "DT_INTER": "Data de internação.",
    "DT_SAIDA": "Data de saída.",
    "DIAS_PERM": "Quantidade de dias de permanência.",
    "PROC_SOLIC": "Procedimento solicitado.",
    "PROC_REA": "Procedimento realizado. Será enriquecido com SIGTAP.",
    "DIAG_PRINC": "Diagnóstico principal, em CID-10.",
    "SEXO": "Sexo do paciente conforme codificação do SIH/SUS.",
    "IDADE": "Idade do paciente.",
    "COD_IDADE": "Unidade/codificação da idade.",
    "MORTE": "Indicador de óbito hospitalar.",
    "VAL_TOT": "Valor total aprovado/pago pelo SUS.",
    "VAL_SH": "Valor de serviços hospitalares.",
    "VAL_SP": "Valor de serviços profissionais.",
    "VAL_UTI": "Valor relacionado à UTI.",
    "COMPLEX": "Complexidade do procedimento/atendimento.",
    "CAR_INT": "Caráter da internação.",
    "ESPEC": "Especialidade associada ao atendimento/leito.",
}

dicionario_df = pd.DataFrame([
    {"campo": campo, "descricao": DICIONARIO_CAMPOS.get(campo, "Campo presente na base SIH/SUS.")}
    for campo in df_sih.columns
])

dicionario_df

## 11. Salvamento da base extraída

Para facilitar a etapa de pipeline, a base tratada será salva em formato Parquet.

Esse arquivo pode ser carregado posteriormente em:

- Oracle Database;
- Oracle Autonomous Database;
- Power BI;
- Streamlit;
- Databricks;
- qualquer ferramenta de análise com suporte a Parquet.

In [ ]:
arquivo_saida = DIRETORIO_TRATADO / f"sih_rd_{UF}_{ANO}_{'-'.join(str(m).zfill(2) for m in MESES)}.parquet"

df_sih.to_parquet(arquivo_saida, index=False)

print(f"Arquivo salvo em: {arquivo_saida}")

## 12. Perguntas que conseguimos responder com SIH/SUS

Com a base extraída, conseguimos responder perguntas ligadas à **demanda hospitalar**.

As perguntas mais fortes para o projeto são:

1. Quais municípios têm maior volume de internações?
2. Quais municípios tiveram maior crescimento de internações ao longo dos meses?
3. Quais hospitais, identificados por CNES, concentram mais internações?
4. Quais hospitais apresentam maior permanência média?
5. Quais procedimentos mais pressionam a rede hospitalar?
6. Quais diagnósticos principais aparecem com maior frequência?
7. Qual é o valor total aprovado/pago por município, hospital ou procedimento?
8. Quais municípios têm maior taxa de óbito hospitalar?
9. Qual perfil etário concentra maior volume de internações?
10. Como a demanda hospitalar varia por sexo, idade, município e período?

Quando o SIH/SUS for cruzado com CNES e IBGE, novas perguntas passam a ser possíveis:

- Quais hospitais apresentam maior pressão entre internações e leitos disponíveis?
- Quais municípios têm mais internações por 10 mil habitantes?
- Quais regiões têm permanência média acima da média estadual?
- Quais regiões combinam crescimento de internações, poucos leitos e alta permanência?

## 13. Exemplo 1 — Volume de internações por município de residência

Esta análise mostra de onde vêm os pacientes internados.

No dashboard, esse indicador ajuda a identificar municípios com maior demanda hospitalar.

In [ ]:
def contar_internacoes(grupo: pd.DataFrame, coluna_id: str = "N_AIH") -> int:
    """Conta internações usando N_AIH quando disponível; caso contrário, conta linhas."""
    if coluna_id in grupo.columns:
        return grupo[coluna_id].nunique(dropna=True)
    return len(grupo)


if "MUNIC_RES" in df_sih.columns:
    internacoes_municipio_res = (
        df_sih
        .groupby("MUNIC_RES")
        .agg(
            internacoes=("N_AIH", "nunique") if "N_AIH" in df_sih.columns else ("MUNIC_RES", "size"),
            permanencia_media=("DIAS_PERM", "mean") if "DIAS_PERM" in df_sih.columns else ("MUNIC_RES", "size"),
            valor_total=("VAL_TOT", "sum") if "VAL_TOT" in df_sih.columns else ("MUNIC_RES", "size"),
            obitos=("MORTE", "sum") if "MORTE" in df_sih.columns else ("MUNIC_RES", "size"),
        )
        .reset_index()
        .sort_values("internacoes", ascending=False)
    )

    internacoes_municipio_res.head(20)
else:
    print("Campo MUNIC_RES não encontrado neste recorte.")

## 14. Exemplo 2 — Ranking de hospitais por internações e permanência média

O código CNES será a chave para, depois, trazer o nome do estabelecimento e sua capacidade instalada.

Neste momento, o SIH/SUS já permite identificar quais estabelecimentos concentram maior volume de internações e maior permanência.

In [ ]:
if "CNES" in df_sih.columns:
    ranking_hospitais = (
        df_sih
        .groupby("CNES")
        .agg(
            internacoes=("N_AIH", "nunique") if "N_AIH" in df_sih.columns else ("CNES", "size"),
            permanencia_media=("DIAS_PERM", "mean") if "DIAS_PERM" in df_sih.columns else ("CNES", "size"),
            valor_total=("VAL_TOT", "sum") if "VAL_TOT" in df_sih.columns else ("CNES", "size"),
            obitos=("MORTE", "sum") if "MORTE" in df_sih.columns else ("CNES", "size"),
        )
        .reset_index()
    )

    if "internacoes" in ranking_hospitais.columns and "obitos" in ranking_hospitais.columns:
        ranking_hospitais["taxa_obito_hospitalar"] = (
            ranking_hospitais["obitos"] / ranking_hospitais["internacoes"]
        )

    ranking_hospitais.sort_values("internacoes", ascending=False).head(20)
else:
    print("Campo CNES não encontrado neste recorte.")

## 15. Exemplo 3 — Procedimentos que mais pressionam a rede

Essa análise identifica quais procedimentos realizados aparecem com maior frequência, maior permanência ou maior custo.

Na próxima etapa, a base **SIGTAP** deve ser usada para traduzir `PROC_REA` para nome do procedimento, grupo, subgrupo e complexidade.

In [ ]:
if "PROC_REA" in df_sih.columns:
    ranking_procedimentos = (
        df_sih
        .groupby("PROC_REA")
        .agg(
            internacoes=("N_AIH", "nunique") if "N_AIH" in df_sih.columns else ("PROC_REA", "size"),
            permanencia_media=("DIAS_PERM", "mean") if "DIAS_PERM" in df_sih.columns else ("PROC_REA", "size"),
            valor_total=("VAL_TOT", "sum") if "VAL_TOT" in df_sih.columns else ("PROC_REA", "size"),
            obitos=("MORTE", "sum") if "MORTE" in df_sih.columns else ("PROC_REA", "size"),
        )
        .reset_index()
        .sort_values("internacoes", ascending=False)
    )

    ranking_procedimentos.head(20)
else:
    print("Campo PROC_REA não encontrado neste recorte.")

## 16. Exemplo 4 — Diagnósticos principais mais frequentes

O campo `DIAG_PRINC` permite analisar o perfil de demanda hospitalar por CID-10.

Isso ajuda a responder quais tipos de condição clínica estão pressionando mais o sistema.

In [ ]:
if "DIAG_PRINC" in df_sih.columns:
    ranking_diagnosticos = (
        df_sih
        .groupby("DIAG_PRINC")
        .agg(
            internacoes=("N_AIH", "nunique") if "N_AIH" in df_sih.columns else ("DIAG_PRINC", "size"),
            permanencia_media=("DIAS_PERM", "mean") if "DIAS_PERM" in df_sih.columns else ("DIAG_PRINC", "size"),
            valor_total=("VAL_TOT", "sum") if "VAL_TOT" in df_sih.columns else ("DIAG_PRINC", "size"),
            obitos=("MORTE", "sum") if "MORTE" in df_sih.columns else ("DIAG_PRINC", "size"),
        )
        .reset_index()
        .sort_values("internacoes", ascending=False)
    )

    ranking_diagnosticos.head(20)
else:
    print("Campo DIAG_PRINC não encontrado neste recorte.")

## 17. Exemplo 5 — Evolução mensal de internações

Essa análise exige mais de um mês de dados em `MESES`.

Ela responde onde a demanda está crescendo ao longo do tempo.

In [ ]:
if "COMPETENCIA" in df_sih.columns:
    evolucao_mensal = (
        df_sih
        .groupby("COMPETENCIA")
        .agg(
            internacoes=("N_AIH", "nunique") if "N_AIH" in df_sih.columns else ("COMPETENCIA", "size"),
            permanencia_media=("DIAS_PERM", "mean") if "DIAS_PERM" in df_sih.columns else ("COMPETENCIA", "size"),
            valor_total=("VAL_TOT", "sum") if "VAL_TOT" in df_sih.columns else ("COMPETENCIA", "size"),
        )
        .reset_index()
        .sort_values("COMPETENCIA")
    )

    evolucao_mensal["crescimento_pct_internacoes"] = evolucao_mensal["internacoes"].pct_change() * 100

    evolucao_mensal
else:
    print("Campo COMPETENCIA não criado. Verifique ANO_CMPT e MES_CMPT.")

## 18. Exemplo 6 — Indicadores executivos para o dashboard

Abaixo estão KPIs iniciais que podem ser exibidos na tela executiva do MVP.

Esses indicadores não dependem ainda de CNES nem IBGE. A pressão assistencial completa será calculada depois, quando cruzarmos com leitos e população.

In [ ]:
def calcular_kpis_sih(df: pd.DataFrame) -> pd.DataFrame:
    kpis = {}

    if "N_AIH" in df.columns:
        kpis["total_internacoes"] = df["N_AIH"].nunique(dropna=True)
    else:
        kpis["total_internacoes"] = len(df)

    if "DIAS_PERM" in df.columns:
        kpis["permanencia_media"] = df["DIAS_PERM"].mean()

    if "VAL_TOT" in df.columns:
        kpis["valor_total_aprovado"] = df["VAL_TOT"].sum()

    if "MORTE" in df.columns:
        kpis["total_obitos_hospitalares"] = df["MORTE"].sum()
        kpis["taxa_obito_hospitalar"] = kpis["total_obitos_hospitalares"] / kpis["total_internacoes"]

    if "CNES" in df.columns:
        kpis["quantidade_hospitais_cnes"] = df["CNES"].nunique(dropna=True)

    if "MUNIC_RES" in df.columns:
        kpis["quantidade_municipios_residencia"] = df["MUNIC_RES"].nunique(dropna=True)

    return pd.DataFrame([kpis])


kpis_sih = calcular_kpis_sih(df_sih)
kpis_sih

## 19. Como esses dados entram no cálculo de pressão assistencial

Somente com SIH/SUS, calculamos a **demanda hospitalar**.

Depois, com CNES e IBGE, o projeto poderá calcular:

| Indicador | Fontes |
|---|---|
| Internações por leito SUS | SIH/SUS + CNES |
| Internações por 10 mil habitantes | SIH/SUS + IBGE |
| Permanência média por hospital e região | SIH/SUS |
| Crescimento mensal de internações | SIH/SUS |
| Índice de pressão assistencial | SIH/SUS + CNES + IBGE |
| Ranking de hospitais sob pressão | SIH/SUS + CNES |
| Ranking de municípios críticos | SIH/SUS + CNES + IBGE |

Sugestão de fórmula inicial para o projeto:

```text
Índice de Pressão Assistencial =
  peso_1 * internações_por_leito
+ peso_2 * crescimento_percentual_internações
+ peso_3 * permanência_média_normalizada
+ peso_4 * taxa_óbito_hospitalar
```

A fórmula deve ser normalizada para que os indicadores fiquem em escala comparável.

## 20. Próximos passos

Depois de validar este notebook do SIH/SUS, os próximos notebooks recomendados são:

1. **CNES** — para extrair estabelecimentos, leitos, tipo de unidade, serviços e habilitações;
2. **IBGE** — para população municipal e normalização de indicadores;
3. **SIGTAP** — para traduzir os códigos de procedimento;
4. **SIA/SUS** — para expandir a análise para demanda ambulatorial e especialidades;
5. **Integração final** — juntar SIH, CNES, IBGE e SIGTAP em tabelas analíticas para o dashboard e para o Select AI.